# The Curse of Dimensionality & Feature Reduction Lab

Adding features without proportionally expanding training data increases model variance, dilutes sample density, and causes Euclidean distances to concentrate. This lab demonstrates how cross-validation error degrades as dimension $p$ outpaces sample count $n$, benchmarks **Univariate Feature Selection (`SelectKBest`)**, and projects correlated features using **Principal Component Analysis (`PCA`)**.

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.datasets import make_regression
from sklearn.pipeline import Pipeline

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Simulating the Curse: Train vs. CV Error as Dimensions Grow

Hold sample size fixed at $N = 100$ while expanding feature dimensions from $p = 2$ to $p = 150$. Watch training $R^2$ reach $1.000$ (pure memorization) while cross-validation $R^2$ collapses below zero.

In [ ]:
n_samples = 100
cv = KFold(n_splits=5, shuffle=True, random_state=42)
feature_counts = [2, 5, 10, 20, 50, 100, 150]

print(f"{'Features (p)':<14} {'Train R²':<14} {'CV R² (Mean)':<16} {'CV Std Dev':<14} {'Curse Status'}")
print("-" * 70)

for p in feature_counts:
    X, y = make_regression(n_samples=n_samples, n_features=p, n_informative=min(5, p), noise=10.0, random_state=42)
    m = Ridge(alpha=1.0).fit(X, y)
    tr_r2 = m.score(X, y)
    cv_scores = cross_val_score(Ridge(alpha=1.0), X, y, cv=cv, scoring='r2')
    
    status = "Severe Overfitting (Curse!)" if tr_r2 - cv_scores.mean() > 0.4 else ("Moderate" if tr_r2 - cv_scores.mean() > 0.15 else "Balanced")
    print(f"{p:<14} {tr_r2:<14.3f} {cv_scores.mean():<16.3f} {cv_scores.std():<14.3f} {status}")

## 2. Distance Concentration in High Dimensions

Sample 100 points uniformly inside a unit hypercube $[0, 1]^d$ and compute the ratio of maximum to minimum pairwise distance: $\frac{d_{\max} - d_{\min}}{d_{\min}}$.

In [ ]:
from scipy.spatial.distance import pdist

dims = [2, 5, 10, 50, 100, 500, 1000]
print(f"{'Dimension (d)':<15} {'Min Distance':<15} {'Max Distance':<15} {'Relative Contrast'}")
print("-" * 60)

for d in dims:
    pts = np.random.uniform(0, 1, size=(100, d))
    dists = pdist(pts)
    d_min, d_max = dists.min(), dists.max()
    contrast = (d_max - d_min) / d_min
    print(f"{d:<15} {d_min:<15.3f} {d_max:<15.3f} {contrast:<15.3f}")

print("\nNotice: As d -> 1000, relative contrast plunges toward zero! All points become equidistant.")

## 3. Defense Benchmarks: Feature Selection vs. PCA Projection

Compare raw 30-feature baseline against Univariate Selection (`SelectKBest(k=10)`) and Linear Projection (`PCA(n_components=10)`).

In [ ]:
X_30, y_30 = make_regression(n_samples=100, n_features=30, n_informative=8, noise=15.0, random_state=42)

# 1. Raw Baseline (All 30 Features)
pipe_raw = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
cv_raw = cross_val_score(pipe_raw, X_30, y_30, cv=cv, scoring='r2')

# 2. Feature Selection (Top 10)
pipe_sel = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_regression, k=10)),
    ('model', Ridge(alpha=1.0))
])
cv_sel = cross_val_score(pipe_sel, X_30, y_30, cv=cv, scoring='r2')

# 3. PCA Projection (10 Components)
pipe_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10)),
    ('model', Ridge(alpha=1.0))
])
cv_pca = cross_val_score(pipe_pca, X_30, y_30, cv=cv, scoring='r2')

print(f"All 30 Features:      CV R² = {cv_raw.mean():.3f} ± {cv_raw.std():.3f} (Sparsity penalty)")
print(f"SelectKBest (k=10):   CV R² = {cv_sel.mean():.3f} ± {cv_sel.std():.3f} (Interpretable selection)")
print(f"PCA (n_comp=10):      CV R² = {cv_pca.mean():.3f} ± {cv_pca.std():.3f} (Correlated variance)")